# 04 · Certifiable-by-design detectors (NF-UNSW-NB15-v3 and NF-ToN-IoT-v3)

Notebook 03 failed its pre-registered pass rules: detectors that use the eight inter-arrival and two per-second byte fields cannot be certified, because those fields have no audited formula and become free as soon as any delay is allowed. This notebook tests one bounded follow-up, fixed on 30 Sep 2026 after notebook 03 and before any v2 result.

Detectors (same parameters, roles, early stopping and 1% false-alarm threshold rule as notebook 02):
- **v2a_no_timing** (ablation): the notebook 02 inputs minus the ten fields with no audited formula.
- **v2b_certifiable** (primary, the pass rules apply to it): v2a, with the five size bins replaced by five cumulative counts (packets at least as large as each bin edge) and monotone training. The score may only rise with every field the declared operations can only raise (packets, bytes, durations, longest packet, cumulative counts, client flags and window), and only fall with receiver-side throughput. Three fields can still move either way (smallest packet, minimum IP length, source throughput); the exact certificate handles them.
- **v2c_shaping_monotone**: v2b without those three fields. Every field the operations can change is monotone in the attacker's direction, so undoing an in-budget operation can never raise the score and every benign verdict is certified by construction. The verifier confirms each one; what is measured is the detection cost.

Certificate, budgets and rules: unchanged from notebook 03 (padding up to 100 bytes per packet, up to 5 dummy packets, up to 1,000 ms delay; 1 second per flow; 5,000 clean benign development flows per dataset and variant; pass if at least 90% CERTIFIED and at most 5% UNRESOLVED). Verifier v2 adds two sound rules: any drop in smallest or longest packet must be paid for with padding bytes, and the original's receiver-side throughput is at least the observed value. If v2b fails, the certificate line stops.

Also reported: the price of certifiability (detection metrics of every variant against the notebook 02 detector), the soundness test for v2b and v2c (v2c should admit no evasion at all), missed attacks and low/high budgets for v2b, and the fields that still block certification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, hashlib, importlib, json, os, sys, time
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
import src.nf3_data as nf
import src.operations as ops
import src.certify as cf
import src.detector as det
for m in (nf, ops, cf, det):
    importlib.reload(m)

TABLES = REPO / 'results' / 'tables'
MODELS_DIR = PROJECT / 'models' / 'nf04'
CACHE = PROJECT / 'cache' / 'nf04'
CACHE.mkdir(parents=True, exist_ok=True)
DATASETS = ['unsw', 'ton']
VARIANTS = ['v2a_no_timing', 'v2b_certifiable', 'v2c_shaping_monotone']
PRIMARY = 'v2b_certifiable'
N_BENIGN, N_SENS, N_MISSED, N_SOUND, TRIES = 5000, 1000, 1000, 2000, 40
TIME_LIMIT = 1.0
SEED = 20260930
FEATS = nf.MODEL_FEATURES
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

def write_config(name, obj):
    text = json.dumps(obj, indent=2, sort_keys=True)
    (REPO / 'configs' / name).write_text(text + '\n')
    return hashlib.sha256(text.encode()).hexdigest()

OPS_SHA = write_config('operations_v2.json', ops.OPERATIONS_CONFIG)
DET_SHA = write_config('detector_v2.json', det.VARIANTS_CONFIG)
V1 = json.loads((TABLES / 'nf02_models.json').read_text())
MACHINE = cf.machine()
VERSIONS = {'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__, 'xgboost': xgb.__version__}
print('machine:', MACHINE)
print('versions:', VERSIONS)
for v in VARIANTS:
    f = det.variant_features(FEATS, v)
    print(f'{v}: {len(f)} inputs, {sum(1 for c in (det.monotone(f, v) or []) if c)} monotone')

## Train the three detectors (detector_train; early stopping and threshold on development)

In [ ]:
D, METRICS, FAMILIES, RECORDS = {}, [], [], {}
for ds in DATASETS:
    cols = FEATS + ['row_idx', 'y', 'family', 'multiplicity', 'role']
    df = nf.load_split(PROJECT, ds, roles=['detector_train', 'development'], columns=cols)
    tr, dv = df[df['role'] == 'detector_train'], df[df['role'] == 'development'].reset_index(drop=True)
    y_tr, y_dv = tr['y'].to_numpy(), dv['y'].to_numpy()
    for v in VARIANTS:
        feats, ctx = det.variant_features(FEATS, v), det.context_features(FEATS, v)
        mono = det.monotone(feats, v)
        Xtr = det.variant_frame(tr[FEATS], FEATS, v)
        Xdv = det.variant_frame(dv[FEATS], FEATS, v)
        booster = det.fit(Xtr, y_tr, Xdv, y_dv, monotone_constraints=mono)
        score = det.predict(booster, Xdv)
        thr = det.threshold_at_fpr(score[y_dv == 0])
        m_rec, f_rec = det.metrics(y_dv, score, thr, family=dv['family'].to_numpy())
        m_w, f_w = det.metrics(y_dv, score, thr, weight=dv['multiplicity'].to_numpy(), family=dv['family'].to_numpy())
        ens = cf.Ensemble(booster, feats)
        viol = det.monotone_violations(ens, Xtr.to_numpy(np.float32), feats, mono) if mono else None
        path = MODELS_DIR / f'{ds}_{v}_xgb.json'
        sha = det.save(booster, path)
        ctx_tr = det.context_frame(tr[FEATS].iloc[:400_000], FEATS, v)
        integer = ops.integer_features(ctx_tr, ctx)
        D[(ds, v)] = {'ens': ens, 'thr': thr, 'feats': feats, 'ctx': ctx, 'integer': integer, 'score': score,
                      'Xctx': det.context_frame(dv[FEATS], FEATS, v).to_numpy(np.float32),
                      'base': dv[FEATS], 'y': y_dv, 'row_idx': dv['row_idx'].to_numpy()}
        RECORDS[f'{ds}/{v}'] = {'model_path': str(path.relative_to(PROJECT)), 'model_sha256': sha, 'threshold': thr,
                                'features': feats, 'monotone_violation_share': viol, **det.tree_stats(booster)}
        METRICS += [{'dataset': ds, 'variant': v, 'weighting': 'distinct records', **m_rec},
                    {'dataset': ds, 'variant': v, 'weighting': 'multiplicity', **m_w}]
        FAMILIES += [f_rec.assign(dataset=ds, variant=v, weighting='distinct records'),
                     f_w.assign(dataset=ds, variant=v, weighting='multiplicity')]
        if mono:
            assert viol == 0, f'{ds}/{v}: the trained model breaks its monotone constraints'
        print(f"{ds}/{v}: AUROC {m_rec['auroc']:.4f} | recall at 1% false alarms {m_rec['recall']:.4f} | "
              f"trees {RECORDS[f'{ds}/{v}']['trees']} | monotone violations {viol}")
        del Xtr, ctx_tr
        gc.collect()
    del df, tr
    gc.collect()

## The price of certifiability: every variant against the notebook 02 detector

In [ ]:
v1m = pd.read_csv(TABLES / 'nf02_detector_metrics.csv').assign(variant='v1 (notebook 02)')
price = pd.concat([v1m[v1m['dataset'].isin(DATASETS)], pd.DataFrame(METRICS)], ignore_index=True)
price = price[['dataset', 'variant', 'weighting', 'auroc', 'pr_auc', 'recall', 'false_alarm_rate', 'precision']]
display(price.sort_values(['dataset', 'weighting', 'variant']))
fam = pd.concat(FAMILIES, ignore_index=True)
display(fam[fam['weighting'] == 'distinct records'].pivot_table(index=['dataset', 'family'], columns='variant',
                                                                 values='flag_rate').round(4))

## Certificates on 5,000 clean benign flows per dataset and variant (primary budget)

In [ ]:
RESULTS, ROWS = {}, []
rng = np.random.default_rng(SEED)
PICK = {}
for (ds, v), st in D.items():
    benign_verdict = st['score'] < st['thr']
    pick = lambda mask, n: np.sort(rng.permutation(np.flatnonzero(mask))[:n])
    PICK[(ds, v)] = {'clean': pick(benign_verdict & (st['y'] == 0), N_BENIGN),
                     'missed': pick(benign_verdict & (st['y'] == 1), N_MISSED),
                     'flagged': pick(~benign_verdict & (st['y'] == 1), N_SOUND)}

def certify(ds, v, which, budget, n=None):
    st = D[(ds, v)]
    pos = PICK[(ds, v)][which][:n] if n else PICK[(ds, v)][which]
    V = cf.Verifier(st['ens'], st['thr'], st['integer'], ops.BUDGETS[budget], context=st['ctx'])
    res = cf.run(V, st['Xctx'][pos], st['row_idx'][pos], CACHE / f'{ds}_{v}_{which}_{budget}.parquet', TIME_LIMIT)
    RESULTS[(ds, v, which, budget)] = res
    row = cf.summarise(res, {'dataset': ds, 'variant': v, 'flows_tested': which, 'budget': budget})
    ROWS.append(row)
    return row

for ds in DATASETS:
    for v in VARIANTS:
        r = certify(ds, v, 'clean', 'primary')
        print(f"{ds}/{v}: certified {r['certified_share']:.4f} | witness {r['witness_share']:.4f} | "
              f"unresolved {r['unresolved_share']:.4f} | median {r['median_seconds']:.3f}s")

## Soundness test: in-budget evasions must never be certified (v2c should admit none)

In [ ]:
SOUND = []
for ds in DATASETS:
    for v in (PRIMARY, 'v2c_shaping_monotone'):
        st = D[(ds, v)]
        di = [st['ctx'].index(f) for f in st['feats']]
        m_thr = cf.logit(st['thr'])
        r2 = np.random.default_rng(SEED + 1)
        ev_rows, ev_ids = [], []
        flagged = PICK[(ds, v)]['flagged']
        for pos in flagged:
            z = ops.add_cumulative(st['base'].iloc[pos].to_dict())
            tries = [ops.add_cumulative(ops.apply_operations(z, ops.BUDGETS['primary'], r2)) for _ in range(TRIES)]
            cand = np.stack([np.array([t[f] for f in st['ctx']], dtype=np.float32) for t in tries])
            m = st['ens'].margin(cand[:, di])
            j = int(np.argmin(m))
            if m[j] < m_thr:
                ev_rows.append(cand[j])
                ev_ids.append(int(st['row_idx'][pos]))
        row = {'dataset': ds, 'variant': v, 'flagged_attacked': len(flagged), 'evasions': len(ev_ids)}
        if ev_ids:
            V = cf.Verifier(st['ens'], st['thr'], st['integer'], ops.BUDGETS['primary'], context=st['ctx'])
            res = cf.run(V, np.stack(ev_rows), ev_ids, CACHE / f'{ds}_{v}_soundness.parquet', TIME_LIMIT)
            row.update(cf.summarise(res, {}))
            assert row['certified'] == 0, f'{ds}/{v}: SOUNDNESS FAILURE, an in-budget evasion was certified'
        SOUND.append(row)
        print(row)
display(pd.DataFrame(SOUND))

## v2b: missed attacks, and low and high budgets on the first 1,000 clean benign flows (reported, not scored)

In [ ]:
for ds in DATASETS:
    if len(PICK[(ds, PRIMARY)]['missed']):
        print(ds, 'missed attacks', json.dumps(certify(ds, PRIMARY, 'missed', 'primary')))
    for budget in ('low', 'high'):
        print(ds, budget, json.dumps(certify(ds, PRIMARY, 'clean', budget, N_SENS)))

## What still blocks certification in v2b

For every WITNESS, the fields that differ between the observed flow and its flagged original show what the attacker would have had to change.

In [ ]:
BLOCK, EXAMPLES = [], []
for ds in DATASETS:
    res = RESULTS[(ds, PRIMARY, 'clean', 'primary')]
    w = res[res['status'] == 'WITNESS']
    counts = {}
    for c in w['changed']:
        for f in json.loads(c):
            counts[f] = counts.get(f, 0) + 1
    for f, n in sorted(counts.items(), key=lambda t: -t[1]):
        BLOCK.append({'dataset': ds, 'feature': f, 'witnesses_changing_it': n, 'share_of_witnesses': n / max(len(w), 1)})
    for _, r in w.sort_values('n_changed').head(3).iterrows():
        EXAMPLES.append({'dataset': ds, 'row_idx': r['row_idx'], 'fields_changed': r['n_changed'],
                         'dummy_pkts': r.get('op_dummy_pkts'), 'padding_bytes': r.get('op_padding_bytes'),
                         'timing_changed': r.get('op_timing_changed'), 'observed -> flagged original': r['changed']})
block = pd.DataFrame(BLOCK, columns=['dataset', 'feature', 'witnesses_changing_it', 'share_of_witnesses'])
for ds in DATASETS:
    display(block[block['dataset'] == ds].head(12))
display(pd.DataFrame(EXAMPLES))

## Decision

In [ ]:
summary = pd.DataFrame(ROWS)
clean = summary[(summary['flows_tested'] == 'clean') & (summary['budget'] == 'primary')].copy()
clean['rule1_certified_ge_90pct'] = clean['certified_share'] >= 0.90
clean['rule2_unresolved_le_5pct'] = clean['unresolved_share'] <= 0.05
clean['passes_rules_1_and_2'] = clean['rule1_certified_ge_90pct'] & clean['rule2_unresolved_le_5pct']
dist = price[price['weighting'] == 'distinct records'][['dataset', 'variant', 'auroc', 'recall']]
decision = clean.merge(dist, on=['dataset', 'variant'], how='left')
display(decision[['dataset', 'variant', 'flows', 'certified_share', 'witness_share', 'unresolved_share', 'median_seconds',
                  'auroc', 'recall', 'rule1_certified_ge_90pct', 'rule2_unresolved_le_5pct', 'passes_rules_1_and_2']])
summary.to_csv(TABLES / 'nf04_certificate_summary.csv', index=False)
price.to_csv(TABLES / 'nf04_price_of_certifiability.csv', index=False)
fam.to_csv(TABLES / 'nf04_family_flag_rates.csv', index=False)
pd.DataFrame(SOUND).to_csv(TABLES / 'nf04_soundness.csv', index=False)
block.to_csv(TABLES / 'nf04_blocking_fields.csv', index=False)
pd.DataFrame(EXAMPLES).to_csv(TABLES / 'nf04_witness_examples.csv', index=False)
for (ds, v, which, budget), res in RESULTS.items():
    res.to_csv(TABLES / f'nf04_flows_{ds}_{v}_{which}_{budget}.csv.gz', index=False)
prim = decision[decision['variant'] == PRIMARY]
out = {'operations_config_sha256': OPS_SHA, 'detector_config_sha256': DET_SHA, 'machine': MACHINE, 'versions': VERSIONS,
       'time_limit_seconds': TIME_LIMIT, 'seed': SEED, 'models': RECORDS,
       'primary_variant': PRIMARY,
       'primary_passes': {r['dataset']: bool(r['passes_rules_1_and_2']) for _, r in prim.iterrows()},
       'certified_share': {f"{r['dataset']}/{r['variant']}": r['certified_share'] for _, r in decision.iterrows()},
       'auroc': {f"{r['dataset']}/{r['variant']}": r['auroc'] for _, r in decision.iterrows()}}
(TABLES / 'nf04_summary.json').write_text(json.dumps(out, indent=2, default=float) + '\n')
print(json.dumps({k: out[k] for k in ('primary_passes', 'certified_share', 'auroc')}, indent=2, default=float))

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 04: certifiable-by-design detectors (v2a without timing fields, v2b with cumulative size counts and monotone training in the attacker's direction, v2c shaping-monotone) with the unchanged certificate, budgets and pass rules; verifier v2 with byte accounting and the receiver-throughput floor; price of certifiability against the notebook 02 detectors; soundness tests; decision on v2b"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)